## A/B Testing Toolkit ##

### Two-Sample Comparison: Confidence Intervals for Difference in Means

This notebook demonstrates how to compare two groups using confidence intervals and hypothesis testing, as commonly used in A/B testing.

The goal is to estimate the difference between two group means and determine whether the observed difference is statistically significant.

This notebook provides reusable functions for:
- Comparing two groups (means)
- Computing confidence intervals
- Performing hypothesis tests (one-sided / two-sided)
- Supporting decision-making in A/B testing scenarios

        Parameters:
            - mean1, mean2: group means
            - std1, std2: standard deviations
            - n1, n2: sample sizes
            - confidence: confidence level (default 0.95)

        Returns:
            - difference, lower bound, upper bound
      
### Assumptions

- Observations are independent  
- Samples are approximately normally distributed (or large enough for CLT)  
- Variances may be 
    - equal variance (pooled)
    - inequal variance (Welch’s method)

### Business Interpretation

- If the confidence interval does not include zero, this suggests a statistically significant difference between groups.  
- In an A/B testing context, this would support deploying the better-performing variant.

### Quick Start

Run the notebook using a simulated dataset to see a full workflow without interaction.

        USE_DEFAULTS = False

In [85]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import glob,os
USE_DEFAULTS = False

## Input format option

You can either: 

 - Enter the sample size, mean and standard deviation of each dataset [s]

 - Otherwise these will be calculated from the input file

In [86]:
data = pd.DataFrame() 
col1, col2 = "Sample 1", "Sample 2"
m1, s1, m2, s2 = 0, 0, 0, 0 # Initialize these too

if USE_DEFAULTS:
    print("Running with default sample data...")
    # Create dummy data so the functions have something to process
    col1, col2 = "Sample A", "Sample B"
    n1, m1, s1 = 100, 50.0, 10.0
    n2, m2, s2 = 100, 55.0, 12.0
    
    data = pd.DataFrame({
        col1: np.random.normal(m1, s1, 100),
        col2: np.random.normal(m2, s2, 100)})
    form1 = 'f'
    infile = 'Simulated test data'
    
else:
    form1 = input("Input from (s)tatistics or (f)ile? ")
    if form1.lower() == "s":
        print("Sample 1...")
        n1 = int(input("Size of Sample 1: ")); 
        m1 = float(input("Mean of Sample 1: ")); 
        s1 = float(input("Standard deviation of Sample 1: "))
        print("Sample 2...")
        n2 = int(input("Size of Sample 2: ")); 
        m2 = float(input("Mean of Sample: ")); 
        s2 = float(input("Standard deviation of Sample 2: "))     
    else:
        print("Choose input file")
    dat = glob.glob('*.dat'); print(dat)
    csv = glob.glob('*.csv'); print(csv)
    infile = str(input("Data with the two samples to compare [e.g. apples.dat]? "))
    check_file = os.path.isfile(infile)
    
    while check_file == False:
        infile = str(input("Data with the two samples to compare [e.g. apples.dat]? "))
        check_file = os.path.isfile(infile)

    #-------READING IN DATA AND CLEANING----------
    df = pd.read_csv(infile, sep=None, engine='python',comment='#')
    df.style.hide(axis='index'); print(df) 
    cols = str(input('Will use the 1st and 2nd columns by default, change these [y/n]? '))
    if cols.lower() == "y" :
        col1 = str(input('Column 1 name: ')); 
        col2 = str(input('Column 2 name: ')); 
    else:
        col1 = df.columns[0]; col2 = df.columns[1] 
        print(df.describe())
    df = df.replace('NaN', np.nan)
    zeroes = str(input("Replace 0s or blanks with NaNs [y/n]? " ))
    if zeroes == "y" or zeroes == "Y":
        df = df.replace(0, np.nan)

    nans1 = df[df[col1] != df[col1]]
    nans2 = df[df[col2] != df[col2]] 
    
    m1 = np.mean(df[col1]); n1 = len(df[col1]) - len(nans1);
    s1 = np.std(df[col1],ddof=1)
    SE1 = s1/(float(n1)**0.5) # SAMPLE SD

    m2 = np.mean(df[col2]); n2 = len(df[col2])-len(nans2); #-nans
    s2 = np.std(df[col2],ddof=1) 
    SE2 = s2/(float(n2)**0.5)

    data = df[[col1,col2]]
    #print(data)
        
def hist(dbs):     
    combined = pd.concat([data[col1], data[col2]])
    min_val, max_val = combined.min(), combined.max()
    
    dbw = (max_val - min_val)/dbs
    bins = np.linspace(min_val, max_val, dbs + 1)

    font = 12
    plt.rcParams.update({'font.size': font})
    plt.figure(figsize=(6,4))
    ax = plt.gca()
    plt.setp(ax.spines.values(),linewidth=2)
    ax.tick_params(direction='in', pad = 7,length=6, width=1.5, which='major',right=True,top=True)
    ax.tick_params(direction='in', pad = 7,length=3, width=1.5, which='minor',right=True,top=True)
    
    label1 = "%s: $\mu = %1.2f, \sigma = %1.2f$" %(col1, m1, s1)
    ax.hist(data[col1], bins=bins, color="w", edgecolor='k', lw=2, label=label1)     

    label2 = "%s: $\mu = %1.2f, \sigma = %1.2f$" %(col2, m2, s2)
    ax.hist(data[col2], bins=bins, color="r", edgecolor='r', lw=2, label=label2, alpha=0.6)
    
    plt.ylabel('Number'); plt.xlabel('Value')
    ax.legend(fontsize = 0.8*font,loc='best')
    plt.show()

Input from (s)tatistics or (f)ile? f
Choose input file
['apples.dat', 'Mg_levels.dat', 'salaries.dat']
['heights.csv']
Data with the two samples to compare [e.g. apples.dat]? Mg_levels.dat
   Patient  Before  After  Difference
0        1     2.0    1.7        -0.3
1        2     1.4    1.7         0.3
2        3     1.3    1.8         0.5
3        4     1.1    1.3         0.2
4        5     1.8    1.7        -0.1
5        6     1.6    1.5        -0.1
6        7     1.5    1.6         0.1
7        8     0.7    1.7         1.0
8        9     0.9    1.7         0.8
9       10     1.5    2.4         0.9
Will use the 1st and 2nd columns by default, change these [y/n]? y
Column 1 name: Before
Column 2 name: After
Replace 0s or blanks with NaNs [y/n]? n


In [87]:
import ipywidgets as widgets

dd_dbs = widgets.Dropdown(options= [2,5,10,20,50,100],value=10,
                            description='No. of bins:',
                            layout={'width': '20%'},  disabled=False)

current_mode = form1.lower() if 'form1' in locals() else 'f'


if USE_DEFAULTS or current_mode != "s":
    out = widgets.interactive_output(hist, {'dbs': dd_dbs})
    ui = widgets.HBox([dd_dbs]) 
    display(ui, out)
else:
    hist(dbs=dd_dbs.value)

Output()

### Functions for determining confidence intervals

Will employ either t- or z-statistics depending on the sample size, which is tuneable in the dropdown below.

In [103]:
def stats(con,sided,equ,cut):
    from scipy.stats import norm
    p = 1-con/100
    from IPython.display import HTML, clear_output
    
    clear_output(wait=True) 
    if sided == "One-tailed":
        p = p; 
    else:
        p = p/2
    alpha = 0.5-p
    
    m = m1 - m2
    SE1 = s1/(float(n1)**0.5)
    SE2 = s2/(float(n2)**0.5) # FOR PLOT
    
    def z_stats(con):
        pooled_var = s1**2/n1 + s2**2/n2
        pooled_SD = pooled_var**0.5
        pooled_SE = pooled_SD*(1/n1 + 1/n2)**0.5
        Z = norm.ppf(1-p,loc=0,scale=1) 
        CI = Z*pooled_SD

        display(HTML("At %1.4f%% confidence the z-value is %1.3f giving a difference in the means of %1.2f +/- %1.2f (%1.2f to %1.2f)" %(con,Z,m,CI,m-CI,m+CI)))
        
        if ((m-CI) * (m+CI)) >= 0:
            display(HTML("Range does not pass through zero so we reject the null hypothesis (the result is significant at %1.4f%% confidence)" %(con)))
        else:
            display(HTML("Range passes through zero so we cannot reject the null hypothesis (the result is not significant at %1.4f%% confidence)" %(con)))
         
        CI_1 = Z*SE1; CI_2 = Z*SE2
        
        return CI_1,CI_2,CI
    
    def t_stats(con):
        npts = 100000
        xi = 0.0; yi = 0;xf = 100; # SHOULD BE INFINITY
        def gamma_f(value): # Gamma function
            x = [xi]; y = [yi]; gamma =0
            for i in range(1,npts):
                dx = (xf-xi)/npts           
                x = i*dx
                y = x**(value-1)*np.exp(-x)
                area =y*dx
                gamma = gamma + area
            return gamma
        
        var_ratio = s1**2/s2**2
        display(HTML("Variance ratio is %1.2f, can assume equal if between 0.5 and 2\n" %(var_ratio)))
        display(HTML("----------------------------------------------------------------------------"))
        
        if equ == "y" or equ == "Y":
            dof = n1 + n2 -2
            pooled_var = ((n1 - 1)*s1**2 + (n2 - 1)*s2**2)/(n1 + n2 -2)
            pooled_SD = pooled_var**0.5
            pooled_SE = pooled_SD*(1/n1 + 1/n2)**0.5; #print(pooled_SD,pooled_SE) #OKAY SO FAR

        else:
            A = s1**2/n1; B =  s2**2/n2
            dof = (A+B)**2/(A**2/(n1-1) + B**2/(n2-1))
            pooled_SE = (s1**2/n1 + s2**2/n2)**0.5;

        def T_int(dof):
            n = dof+1
            gamma_num = gamma_f(float(n)/2)
            gamma_den = gamma_f(float(dof)/2)
            stand =  gamma_num/(((np.pi*dof)**0.5)*gamma_den)

            npts = 100000; 
            xf = 5;x = [xi];yy = [yi]; total =0;y_total = 0; total = 0; area = 0; j =0
            dx = (xf-xi)/npts
            for i in range(npts):
                x = i*dx
                y = stand*(1+ x**2/dof)**float(-n/2)
                while(total < alpha):
                    y_total = stand*(1+ (j*dx)**2/dof)**float(-n/2)
                    area = y_total*dx
                    total = total + area; 
                    j = j+1
            T = dx*j; 
            return T
        T = T_int(dof)
        CI = T*pooled_SE
        #### INDIVIDUAL VALUES FOR PLOT BELOW###
        dof1 = n1 -1; T1 = T_int(dof1); CI_1 = T1*SE1
        dof2 = n2 -1; T2 = T_int(dof2); CI_2 = T2*SE2
        #######################################
    
        display(HTML("At %1.4f%% confidence (%1.2f DoFs) the t-value is %1.3f giving a difference in the means of %1.2f +/- %1.2f (%1.2f to %1.2f)" %(con,dof,T,m,CI,m-CI,m+CI)))
        
        if ((m-CI)*(m+CI)) >= 0:
            display(HTML("Range does not pass through zero so we reject the null hypothesis (the result is significant at %1.4f%% confidence)" %(con)))
        else:
            display(HTML("Range passes through zero so we cannot reject the null hypothesis (the result is not significant at %1.4f%% confidence)" %(con)))
        return CI_1,CI_2,CI
    
    # OUPUT
    if form1.lower() != "s": 
        col1 = data.columns[0]; col2 = data.columns[1]
        display(HTML('\n============================= For %s ============================= ' %(infile)))    
    else:
         col1 = 'Sample 1'; col2 = 'Sample 2'
    display(HTML("%s: n = %d, mean = %1.3f, standard deviation = %1.3f" %(col1,n1,m1,s1)))
    display(HTML("%s: n = %d, mean = %1.3f, standard deviation = %1.3f" %(col2,n2,m2,s2)))
    display(HTML("----------------------------------------------------------------------------"))
    
    if n1 > cut and n2 > cut:
        display(HTML("\nBoth sample sizes > %d so using z-statistics" %(cut)))
        CI_1,CI_2,CI = z_stats(con)
    else:
        display(HTML("\nAt least one sample size < %d so using t-statistics" %(cut)))
        CI_1,CI_2,CI = t_stats(con)
    
    if m-CI > 0:
        print("Summary: %s is significantly greater than %s" %(col1,col2))
    elif m+CI < 0:
        print("Summary: %s is significantly greater than %s" %(col2,col1))
    else:
        print("Summary: there is no statistically significant difference")
              
    # SHOW DISTRIBUTIONS 
    font = 12
    plt.rcParams.update({'font.size': font})
    plt.figure(figsize = (6,2))
    ax = plt.gca();
    plt.setp(ax.spines.values(),linewidth=2)
    
    y_numeric = [0.3, 0.7]
    y_words = [col1,col2]
    
    ax.errorbar(m1,y_numeric[0], xerr=CI_1, fmt='o', color='r', capsize=5, label= "$%1.3f\pm%1.3g$" %(m1,CI_1))
    ax.errorbar(m2, y_numeric[1], xerr=CI_2, fmt='o', color='g', capsize=5, label= "$%1.3f\pm%1.3g$" %(m2,CI_2))
    
    #ax.errorbar(m1,y_numeric[0], xerr=s1/np.sqrt(n1), fmt='o', color='k', capsize=1, label= "$%1.3f\pm%1.3g$" %(m1,s1/np.sqrt(n1)))
    #ax.errorbar(m2, y_numeric[1], xerr=s2/np.sqrt(n2), fmt='o', color='k', capsize=1, label= "$%1.3f\pm%1.3g$" %(m2,s2/np.sqrt(n2)))

    ax.set_ylim(0,1)
    ax.set_xlabel("Mean values (%1.2f%% confidence)" %(con));
    plt.yticks(y_numeric, y_words)
    ax.legend(fontsize = 0.8*font,loc='best')
    plt.tight_layout()
    plt.show()   

## Interactive drop down menus for the options

Here you can:
  - Alter the confidence level
  - Run a one- or two-tailed test
  - Change the t- to z-statistic sample size threshold from the default 30
  - If t-statistic is used, choose whether or not to assume equal variances

In [104]:
dd_con = widgets.Dropdown(options= [90,95,99,99.9,99.99,99.999], value=95, 
                        description='Confidence level [%]',
                        style={'description_width': 'initial'},
                        layout={'width': '25%'}, disabled=False)

dd_s = widgets.Dropdown(options= ['One-tailed','Two-tailed'], value='Two-tailed', 
                        description='One or two tailed test:',
                        style={'description_width': 'initial'},
                        layout={'width': '30%'}, disabled=False)

dd_cut = widgets.Dropdown(options= [10,20,30,50,100], value=30, 
                        description='Sample sizes for t- to z-distribution theshold:',
                        style={'description_width': 'initial'},
                        layout={'width': '50%'}, disabled=False)

dd_v = widgets.Dropdown(options= ['y','n'], value='y', 
                        description='Assume equal sample variances:',
                        style={'description_width': 'initial'},
                        layout={'width': '35%'}, disabled=False)

out = widgets.interactive_output(stats,{'con':dd_con, 'sided':dd_s,'cut':dd_cut,'equ':dd_v})
ui1 = widgets.HBox([dd_con, dd_s])
ui2 = widgets.HBox([dd_cut,dd_v])
display(ui1,ui2,out)

if USE_DEFAULTS:
    stats(con=dd_con.value, sided=dd_s.value,cut=dd_cut.value,equ=dd_v.value)

Output()